In [1]:
# Section 1: Initial Setup, Warnings & Temporary Directory Creation
import warnings, os, tempfile
warnings.filterwarnings('ignore')

TEMP_DIR = tempfile.mkdtemp()
print('Temporary Directory Created:')
print(TEMP_DIR)

Temporary Directory Created:
C:\Users\chinm\AppData\Local\Temp\tmpbjttdkgo


In [2]:
# Section 2: Custom CSS Styling for Notebook UI Components
from IPython.display import display, HTML

display(HTML("""
<style>
.glass-card{background:rgba(255,255,255,0.08);border-radius:20px;padding:20px;margin:15px 0;backdrop-filter:blur(10px);}
.gradient-text{background:linear-gradient(45deg,#00c6ff,#0072ff);-webkit-background-clip:text;color:transparent;font-weight:bold;}
</style>
"""))

In [3]:
# Section 3: Status Display Utility Function
from IPython.display import display, HTML, clear_output

def show_status(message):
    clear_output(wait=True)
    display(HTML(f"<div class='glass-card'>🤖 {message}</div>"))

In [ ]:
# Section 4: File Upload Widget & UI Controls
import ipywidgets as widgets
from IPython.display import display

video_path = None  # Will be set when a file is uploaded

# --- File upload widget ---
upload_widget = widgets.FileUpload(
    accept='.mp4,.avi,.mov,.mkv',
    multiple=False,
    description='Upload Video',
)

def on_upload_change(change):
    global video_path
    if upload_widget.value:
        # ipywidgets v8+: .value is a tuple of dicts
        uploaded_file = upload_widget.value[0]
        filename = uploaded_file['name']
        content  = uploaded_file['content']
        
        video_path = os.path.join(TEMP_DIR, filename)
        with open(video_path, 'wb') as f:
            f.write(content)
        print(f'✅ Video uploaded: {filename}')

upload_widget.observe(on_upload_change, names='value')

# --- Analyze button ---
analyze_btn = widgets.Button(
    description='▶ Analyze Video',
    button_style='success',
    layout=widgets.Layout(width='200px', height='40px'),
)

# --- Display UI ---
display(widgets.VBox([
    widgets.HTML("<h3>🎥 Interview Analyzer</h3>"),
    upload_widget,
    analyze_btn,
]))

c:\Users\chinm\Desktop\education\project\Interview_analyse\Interview_Professional_Report.pdf

In [5]:
# Section 5: Audio Extraction from Video
import os
from moviepy import VideoFileClip  # Changed import

def extract_audio(video_path):
    audio_path = os.path.join(TEMP_DIR, 'audio.wav')
    video = VideoFileClip(video_path)  # Changed usage
    video.audio.write_audiofile(audio_path, logger=None)
    video.close()
    return audio_path

In [6]:
# Section 6: Video Frame Sampling
import cv2

def sample_frames(video_path, fps=0.5):
    cap = cv2.VideoCapture(video_path)
    original_fps = cap.get(cv2.CAP_PROP_FPS)
    interval = int(original_fps / fps)
    frames, timestamps = [], []
    count = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if count % interval == 0:
            frames.append(frame)
            timestamps.append(count / original_fps)
        count += 1
    cap.release()
    return frames, timestamps

In [7]:
# Section 7: Whisper Model Loading & Audio Transcription
from faster_whisper import WhisperModel

whisper_model = WhisperModel('tiny', device='cpu', compute_type='int8') # should use 'tiny'or 'medium' instead of large of the ram is small or equal to 16 GB

def transcribe_audio(audio_path):
    segments, info = whisper_model.transcribe(audio_path)
    transcript = ' '.join([seg.text for seg in segments])
    duration = info.duration
    total_words = len(transcript.split())
    wpm = (total_words / duration) * 60 if duration > 0 else 0
    return {'transcript': transcript, 'wpm': wpm}

In [8]:
# Section 8: Facial Emotion Analysis Using DeepFace
import cv2
from tqdm.notebook import tqdm
from deepface import DeepFace

def analyze_emotions(frames, timestamps):
    emotions = []
    for idx, frame in enumerate(tqdm(frames)):
        try:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            result = DeepFace.analyze(rgb, actions=['emotion'], enforce_detection=False)
            dominant = result[0]['dominant_emotion']
            emotions.append({'timestamp': timestamps[idx], 'emotion': dominant})
        except:
            emotions.append({'timestamp': timestamps[idx], 'emotion': 'neutral'})
    return emotions

In [9]:
# Section 9: Eye Contact Detection Using Haar Cascades
import cv2
import numpy as np
from tqdm.notebook import tqdm

# Load Haar Cascades
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
eye_cascade  = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_eye.xml')

def detect_eye_contact(frames, timestamps):
    eye_scores = []
    for idx, frame in enumerate(tqdm(frames)):
        gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, 1.3, 5)
        contact = 0
        for (x, y, w, h) in faces:
            roi_gray = gray[y:y+h, x:x+w]
            eyes = eye_cascade.detectMultiScale(roi_gray)
            # If 2 eyes visible -> likely eye contact
            if len(eyes) >= 2:
                contact = 1
        eye_scores.append({'timestamp': timestamps[idx], 'contact': contact})
    avg_contact = np.mean([x['contact'] for x in eye_scores])
    return eye_scores, avg_contact

In [10]:
# Section 10: Filler Word Detection & Ratio Calculation
FILLER_WORDS = ['um', 'uh', 'ah', 'like', 'you know', 'actually', 'basically', 'so', 'well']

def detect_fillers(transcript):
    transcript_lower = transcript.lower()
    count = sum(transcript_lower.count(f) for f in FILLER_WORDS)
    total_words = len(transcript.split())
    filler_ratio = (count / total_words) * 100 if total_words > 0 else 0
    return {'total_fillers': count, 'filler_ratio': filler_ratio}

In [11]:
# Section 11: Sentiment Analysis Pipeline & Voice Confidence Analysis
import numpy as np
import librosa
from transformers import pipeline

sentiment_pipeline = pipeline(
    'sentiment-analysis',
    model='distilbert-base-uncased-finetuned-sst-2-english',
)

def analyze_voice(audio_path):
    try:
        y, sr = librosa.load(audio_path, sr=None)
        rms        = librosa.feature.rms(y=y)[0]
        zcr        = librosa.feature.zero_crossing_rate(y)[0]
        mfccs      = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        pitches, mags = librosa.piptrack(y=y, sr=sr)
        intervals  = librosa.effects.split(y, top_db=30)
        spec_cent  = librosa.feature.spectral_centroid(y=y, sr=sr)[0]

        pitch_vals  = pitches[mags > np.median(mags) + 1e-6]
        pitch_vals  = pitch_vals[pitch_vals > 50]
        pitch_std   = np.std(pitch_vals) if len(pitch_vals) else 0
        speak_ratio = sum(e - s for s, e in intervals) / max(len(y), 1)
        pause_rate  = max(0, len(intervals) - 1) / max(len(y) / sr, 1)
        spec_mean   = np.mean(spec_cent)

        scores = {
            'energy':      min(100, np.mean(rms) * 1000),
            'consistency': max(0, 100 - (np.std(rms) / (np.mean(rms) + 1e-6)) * 100),
            'speaking':    speak_ratio * 100,
            'pauses':      max(0, 100 - pause_rate * 20),
            'pitch':       min(100, pitch_std * 0.4) if 10 <= pitch_std <= 120 else (20 if pitch_std < 10 else 60),
            'expression':  min(100, np.mean(np.std(mfccs, axis=1)) * 3),
            'steadiness':  max(0, 100 - np.std(zcr) * 2000),
            'clarity':     100 if 1000 <= spec_mean <= 3000 else 70 if 500 <= spec_mean <= 4000 else 40,
        }
        weights = [0.15, 0.10, 0.15, 0.10, 0.15, 0.10, 0.10, 0.15]
        score   = round(min(100, max(0, sum(v * w for v, w in zip(scores.values(), weights)))), 1)
        label   = 'High' if score >= 80 else 'Moderate' if score >= 60 else 'Low' if score >= 40 else 'Very Low'
        return {'confidence_score': score, 'confidence_label': f'{label} Confidence', **scores}
    except Exception as e:
        print(f'[analyze_voice] Error: {e}')
        return {'confidence_score': 50.0, 'confidence_label': 'Unknown'}


Device set to use cpu


In [12]:
# Section 12: Basic Score Computation (Confidence & Communication)
import numpy as np

def compute_scores(metrics):
    confidence = (
        metrics['voice_confidence'] * 0.4 +
        (metrics['eye_contact'] * 100) * 0.3 +
        max(0, 100 - metrics['filler_ratio'] * 2) * 0.3
    )
    wpm_score = max(0, 100 - abs(metrics['wpm'] - 140))
    communication = (
        wpm_score * 0.4 +
        max(0, 100 - metrics['filler_ratio'] * 5) * 0.6
    )
    overall = np.mean([confidence, communication])
    return {
        'Confidence':    round(confidence, 1),
        'Communication': round(communication, 1),
        'Overall Score': round(overall, 1)
    }

In [13]:
# Section 13: Main Analysis Pipeline & Orchestration
from IPython.display import display, FileLink, clear_output

def run_analysis(b):
    if video_path is None:
        print('Upload a video first.')
        return

    show_status('Extracting Audio...')
    audio_path = extract_audio(video_path)

    show_status('Sampling Frames...')
    frames, timestamps = sample_frames(video_path)

    show_status('Transcribing Audio...')
    transcript_data = transcribe_audio(audio_path)
    transcript = transcript_data['transcript']

    show_status('Analyzing Emotions...')
    emotions = analyze_emotions(frames, timestamps)

    show_status('Tracking Eye Contact...')
    eye_scores, avg_eye = detect_eye_contact(frames, timestamps)

    show_status('Analyzing Voice...')
    voice_features = analyze_voice(audio_path)

    filler_stats = detect_fillers(transcript)
    sentiment    = sentiment_pipeline(transcript[:512])[0]

    emotion_names    = [e['emotion'] for e in emotions]
    dominant_emotion = max(set(emotion_names), key=emotion_names.count)

    metrics = {
        'wpm':              transcript_data['wpm'],
        'eye_contact':      avg_eye,
        'voice_confidence': voice_features['confidence_score'],
        'filler_ratio':     filler_stats['filler_ratio'],
        'dominant_emotion': dominant_emotion,
    }

    scores = compute_scores(metrics)

    show_status('Running Enhanced Analytics...')
    emotion_data    = analyze_emotion_distribution(emotions)
    speaking_data   = analyze_speaking(transcript, transcript_data['wpm'])
    filler_data     = analyze_fillers_enhanced(transcript)
    content_data    = analyze_content_quality(transcript)
    advanced_scores = compute_advanced_scores(metrics, emotion_data, content_data, filler_data)
    strengths, improvements = generate_strengths_improvements(
        advanced_scores, speaking_data, filler_data, content_data, emotion_data
    )

    duration_sec = timestamps[-1] if timestamps else 0

    show_status('Generating PDF Report...')
    pdf_path = generate_professional_pdf(
        advanced_scores, speaking_data, filler_data, avg_eye,
        emotion_data, content_data, strengths, improvements,
        transcript, duration_sec,
    )

    clear_output(wait=True)
    print(f'✅ Analysis Complete! PDF saved to: {pdf_path}')
    display(FileLink(pdf_path, result_html_prefix='📄 Download Report: '))

analyze_btn.on_click(run_analysis)


In [14]:
# Section 14: Enhanced Emotion Distribution & Stability Analysis
import numpy as np

EMOTION_LABELS = ['neutral', 'happy', 'sad', 'angry', 'surprise', 'fear', 'disgust']

def analyze_emotion_distribution(emotions):
    """
    Compute full emotion distribution and stability score
    from the list of per-frame emotion dicts returned by analyze_emotions().
    """
    counts = {e: 0 for e in EMOTION_LABELS}
    for entry in emotions:
        label = entry['emotion'].lower()
        counts[label if label in counts else 'neutral'] += 1

    total = max(len(emotions), 1)
    distribution = {k: round((v / total) * 100, 1) for k, v in counts.items()}

    positive_pct = distribution.get('happy', 0)
    neutral_pct  = distribution.get('neutral', 0)
    negative_pct = (
        distribution.get('angry', 0) + distribution.get('sad', 0) +
        distribution.get('fear', 0) + distribution.get('disgust', 0)
    )
    stability_score = min(100, max(0,
        (neutral_pct + positive_pct) * 0.8 - negative_pct * 0.5 + 20  # baseline
    ))

    if stability_score >= 80:   stability_label = 'Excellent'
    elif stability_score >= 60: stability_label = 'Good'
    else:                       stability_label = 'Needs Improvement'

    return {
        'distribution':    distribution,
        'stability_score': round(stability_score, 1),
        'stability_label': stability_label,
    }

In [15]:
# Section 15: Speaking Pace Analysis & Classification
def analyze_speaking(transcript, wpm):
    """
    Extend basic WPM with pace classification,
    total words, and a human-readable explanation.
    """
    total_words = len(transcript.split())
    if wpm < 110:
        pace = 'Too Slow'
        pace_explanation = (
            'Your speaking pace is below the ideal range. '
            'Try to pick up speed to keep the interviewer engaged.'
        )
    elif wpm <= 160:
        pace = 'Ideal'
        pace_explanation = (
            'Your speaking pace is in the ideal range (110–160 WPM). '
            'This makes it easy for the interviewer to follow you.'
        )
    else:
        pace = 'Too Fast'
        pace_explanation = (
            'You are speaking faster than the recommended pace. '
            'Slow down slightly so your words are easier to process.'
        )
    return {
        'total_words':      total_words,
        'wpm':              round(wpm, 1),
        'pace':             pace,
        'pace_explanation': pace_explanation,
    }

In [16]:
# Section 16: Enhanced Filler Word Analysis with Extended Word List
import re
from collections import Counter

FILLER_WORDS_LIST = [
    # --- Original Base List ---
    'um', 'uh', 'ah', 'like', 'you know', 'actually', 'basically', 'so', 'well',

    # --- Common Verbal Crutches & Hesitations ---
    'er', 'hmm', 'uh-huh', 'right', 'okay', 'ok', 'yeah', 'mean', 'i mean',

    # --- Discourse Markers & Transition Fillers ---
    'honestly', 'literally', 'seriously', 'obviously', 'essentially', 
    'virtually', 'totally', 'completely', 'fundamentally', 'relatively',

    # --- Mitigators & Softeners (Hedging) ---
    'probably', 'maybe', 'sort of', 'kind of', 'somewhat', 'just', 
    'supposedly', 'presumably', 'perhaps',

    # --- Conversational & Attention-Seeking Phrases ---
    'you see', 'look', 'listen', 'tell you what', 'to be honest', 
    'truth be told', 'at the end of the day', 'for what it\'s worth',
    'believe me', 'mind you', 'by the way', 'in any case',

    # --- Delay Tactics & Reformulation Phrases ---
    'as I said', 'so to speak', 'in a way', 'more or less', 
    'all in all', 'or something', 'or whatever', 'and stuff'
]

def analyze_fillers_enhanced(transcript):
    """
    Per-filler word counts, ratio, and severity level.
    (Keeps existing detect_fillers() results; just adds more detail.)
    """
    text_lower = transcript.lower()
    filler_counts = {}
    for filler in FILLER_WORDS_LIST:
        count = (text_lower.count(filler) if ' ' in filler
                 else len(re.findall(r'\b' + re.escape(filler) + r'\b', text_lower)))
        if count > 0:
            filler_counts[filler] = count

    total_fillers = sum(filler_counts.values())
    total_words   = max(len(transcript.split()), 1)
    filler_ratio  = round((total_fillers / total_words) * 100, 1)

    # Most common fillers sorted descending
    common = sorted(filler_counts.items(), key=lambda x: x[1], reverse=True)[:5]

    severity = ('Excellent' if filler_ratio <= 2 else
                'Good'      if filler_ratio <= 5 else 'Needs Improvement')

    return {
        'total_fillers':  total_fillers,
        'filler_ratio':   filler_ratio,
        'common_fillers': common,
        'severity':       severity,
        'filler_counts':  filler_counts,
    }

In [17]:
# Section 17: Technical Keyword Detection & Content Quality Analysis

TECH_KEYWORDS = [
    # --- Original Base List ---
    'machine learning', 'deep learning', 'tensorflow', 'pytorch',
    'keras', 'opencv', 'python', 'java', 'c++', 'javascript',
    'react', 'node.js', 'django', 'flask', 'fastapi',
    'cnn', 'rnn', 'lstm', 'transformer', 'bert', 'gpt',
    'yolo', 'resnet', 'vgg', 'gan', 'vae',
    'nlp', 'natural language processing', 'computer vision',
    'data science', 'data engineering', 'data analysis',
    'sql', 'nosql', 'mongodb', 'postgresql', 'mysql',
    'docker', 'kubernetes', 'aws', 'azure', 'gcp',
    'git', 'ci/cd', 'devops', 'mlops',
    'api', 'rest', 'graphql', 'microservices',
    'neural network', 'regression', 'classification',
    'clustering', 'reinforcement learning',
    'pandas', 'numpy', 'scikit-learn', 'matplotlib',
    'spark', 'hadoop', 'kafka', 'airflow',
    'accuracy', 'precision', 'recall', 'f1',
    'overfitting', 'underfitting', 'cross-validation',

    # --- Generative AI, LLMs, & Agentic Systems ---
    'mixture of experts', 'moe', 'vision language models', 'vlm', 'diffusion models',
    'state space models', 'ssm', 'mamba architecture', 'autoregressive models',
    'bidirectional transformers', 'retrieval-augmented generation', 'rag', 'graph-rag',
    'corrective rag', 'crag', 'self-rag', 'multimodal llms', 'large audio models',
    'lam', 'diffusion transformers', 'dit', 'sparse transformers', 'flashattention',
    'speculative decoding', 'contrastive learning', 'masked language modeling', 'mlm',
    'causal language modeling', 'clm', 'parameter-efficient fine-tuning', 'peft',
    'low-rank adaptation', 'lora', 'quantized lora', 'qlora', 'p-tuning',
    'prompt tuning', 'prefix tuning', 'reinforcement learning from human feedback', 'rlhf',
    'reinforcement learning from ai feedback', 'rlaif', 'direct preference optimization', 'dpo',
    'odds ratio preference optimization', 'orpo', 'proximal policy optimization', 'ppo',
    'kahneman-tversky optimization', 'kto', 'supervised fine-tuning', 'sft',
    'instruction tuning', 'full parameter fine-tuning', 'chain of thought', 'cot',
    'tree of thoughts', 'tot', 'skeleton-of-thought', 'recommender agents',
    'react framework', 'langchain', 'langgraph', 'llamaindex',
    'autogen', 'crewai', 'semantic kernel', 'haystack',
    'fixie.ai', 'prompt injection mitigation', 'prompt chaining', 'few-shot prompting',
    'zero-shot prompting', 'metaprompting', 'agentic workflows', 'multi-agent systems',
    'tool calling', 'function calling',

    # --- Model Evaluation, Trust, Safety, & Ethics ---
    'lm-eval-harness', 'ragas', 'truelens', 'deepeval',
    'promptfoo', 'bleu score', 'rouge score', 'meteor score',
    'perplexity', 'bertscore', 'exact match', 'em',
    'human-in-the-loop', 'hitl', 'llm-as-a-judge', 'mmlu',
    'gsm8k', 'humaneval', 'mbpp', 'hellaswag',
    'arc', 'ai2 reasoning challenge', 'guardrails ai', 'nemo guardrails',
    'llama guard', 'red teaming', 'adversarial probing', 'jailbreaking mitigation',
    'hallucination detection', 'toxicity scoring', 'bias mitigation', 'fairness metrics',
    'disparate impact', 'equalized odds', 'differential privacy', 'federated learning',
    'homomorphic encryption', 'anonymization pipelines', 'pii masking', 'explainable ai',
    'xai', 'shap', 'lime', 'integrated gradients',
    'model card reporting', 'ai provenance', 'watermarking llm outputs',

    # --- Data Engineering, Data Mesh, & Analytics Infrastructure ---
    'apache iceberg', 'delta lake', 'apache hudi', 'parquet',
    'orc', 'avro', 'apache arrow', 'feather',
    'protocol buffers', 'protobuf', 'thrift', 'object storage',
    'amazon s3', 'google cloud storage', 'gcs', 'azure blob storage',
    'minio', 'ceph', 'hdfs', 'data lakehouse',
    'cloud data warehouse', 'pyspark', 'spark sql', 'spark streaming',
    'apache flink', 'ray data', 'dask', 'databricks',
    'snowflake', 'google bigquery', 'aws redshift', 'clickhouse',
    'duckdb', 'trino', 'presto', 'prefect',
    'dagster', 'mage-ai', 'luigi', 'apache beam',
    'talend', 'matillion', 'fivetran', 'stitch',
    'dbt', 'data build tool', 'data mesh', 'data contract',
    'data lineage', 'data catalog', 'apache atlas', 'amundsen',
    'datahub', 'collibra', 'alation', 'great expectations',
    'monte carlo ai', 'soda core', 'anomalo', 'data quality engine',
    'schema registry', 'schema evolution', 'idempotent pipelines', 'backfilling pipelines',
    'change data capture', 'cdc', 'debezium', 'event-driven architecture',
    'apache pulsar', 'aws kinesis', 'rabbitmq',

    # --- Vector Databases & Information Retrieval ---
    'dense retrieval', 'sparse retrieval', 'hybrid search', 'vector embeddings',
    'text-embedding-3', 'cohere embeddings', 'sentence-transformers', 'cosine similarity',
    'dot product distance', 'euclidean distance', 'l2 distance', 'manhattan distance',
    'l1 distance', 'approximate nearest neighbors', 'ann', 'hnsw',
    'ivfflat', 'product quantization', 'pq', 'locality-sensitive hashing',
    'lsh', 'pinecone', 'milvus', 'qdrant',
    'chromadb', 'weaviate', 'faiss', 'pgvector',
    'elasticsearch', 'opensearch', 'apache solr', 'bm25 algorithm',
    'tf-idf', 'inverted index', 'two-stage retrieval', 'cross-encoders',
    'bi-encoders', 'learning to rank', 'ltr', 'reciprocal rank fusion',
    'rrf', 'query expansion', 'semantic search',

    # --- MLOps, LLMOps, & Feature Stores ---
    'mlflow', 'weights & biases', 'wandb', 'comet ml',
    'neptune.ai', 'dvc', 'data version control', 'clearml',
    'kubeflow', 'zenml', 'metaflow', 'kedro',
    'model registry', 'artifact tracking', 'experiment lineage', 'reproducibility pipeline',
    'feast', 'feature store', 'tecton', 'hopsworks',
    'databricks feature store', 'sagemaker feature store', 'online feature store', 'offline feature store',
    'feature leakage mitigation', 'time-travel queries', 'entity dataframe', 'feature view',
    'streaming ingestion', 'batch ingestion', 'triton inference server', 'torchserve',
    'tensorflow serving', 'seldon core', 'kserve', 'bentoml',
    'vllm', 'tgi', 'text generation inference', 'ollama',
    'arize ai', 'whylabs', 'evidently ai', 'fiddler ai',
    'datadog', 'dynatrace', 'prometheus', 'grafana',
    'data drift detection', 'concept drift detection', 'covariate shift', 'prior probability shift',
    'model degradation tracking',

    # --- AI Infrastructure, Distributed Systems, & Hardware ---
    'nvidia h100', 'nvidia a100', 'nvidia b200', 'blackwell',
    'tensor processing unit', 'tpu', 'language processing unit', 'lpu',
    'groq lpu', 'amd instinct', 'mi300x', 'aws inferentia',
    'aws trainium', 'application-specific integrated circuit', 'asic', 'field-programmable gate array',
    'fpga', 'cuda core', 'tensor core', 'vram allocation',
    'sram optimizer', 'nvlink', 'infiniband', 'pcie gen5',
    'distributed data parallel', 'ddp', 'fully sharded data parallel', 'fsdp',
    'megatron-lm', 'deepspeed', 'horovod', 'pipeline parallelism',
    'tensor parallelism', 'expert parallelism', 'sequence parallelism', 'data parallelism',
    'activation checkpointing', 'gradient accumulation', 'mixed precision training', 'fp16',
    'bf16', 'fp8 training', 'int8 quantization', 'int4 quantization',
    'awq', 'activation-aware weight quantization', 'gptq', 'gguf',
    'bitsandbytes', 'onnx runtime', 'tensorrt', 'openvino',
    'slurm workload manager', 'kubernetes cluster api', 'gpu virtualization', 'mig',
    'multi-instance gpu', 'nccl', 'gloo', 'mpi',
    'message passing interface', 'kubernetes device plugins', 'bare-metal provisioning', 'spot instance orchestration',
    'cluster auto-scaling', 'fault-tolerant training',

    # --- Graph Machine Learning & Advanced ML Systems ---
    'graph convolutional networks', 'gcn', 'graph attention networks', 'gat',
    'graphsage', 'message passing neural networks', 'mpnn', 'relational gcn',
    'r-gcn', 'temporal graph networks', 'tgn', 'heterogeneous graph networks',
    'graph isomorphisms', 'node embeddings', 'edge prediction', 'graph classification',
    'link prediction', 'node2vec', 'deepwalk', 'pyg',
    'pytorch geometric', 'dgl', 'deep graph library', 'networkx',
    'neo4j', 'amazon neptune', 'memgraph', 'compiler optimization',
    'tvm', 'apache tvm', 'xla', 'accelerated linear algebra',
    'torch.compile', 'triton compiler', 'kernel fusion', 'memory bandwidth optimization',
    'cache locality', 'asynchronous computation', 'graph compilation', 'static computation graphs',
    'dynamic computation graphs', 'operator serialization', 'custom cuda kernels',

    # --- Computer Vision, Edge AI, Spatial Computing & Robotics ---
    'object detection', 'semantic segmentation', 'instance segmentation', 'panoptic segmentation',
    'yolov8', 'yolov9', 'yolov10', 'segment anything model',
    'sam', 'dino-v2', 'vision transformers', 'vit',
    'swin transformer', 'optical flow', 'image registration', 'edge detection',
    'pose estimation', 'facial recognition', 'action recognition', '3d object detection',
    'point clouds', 'open3d', 'pcl', 'point cloud library',
    'nerf', 'neural radiance fields', '3d gaussian splatting', 'slam',
    'simultaneous localization and mapping', 'visual-inertial odometry', 'vio', 'depth estimation',
    'structure from motion', 'sfm', 'mesh generation', 'ray tracing',
    'spatial anchors', 'hand tracking algorithms', 'eye tracking models', 'occlusion culling',
    'unity sentinel', 'unreal engine blueprinting', 'tinyml', 'tensorflow lite',
    'pytorch mobile', 'coreml', 'nnapi', 'quantization-aware training',
    'qat', 'post-training quantization', 'ptq', 'knowledge distillation',
    'network pruning', 'hardware-aware neural architecture search', 'nas', 'edge impulse',
    'jetson nano', 'jetson orin', 'raspberry pi compute module', 'microcontrollers',
    'cortex-m', 'low-latency inferencing', 'ros', 'robot operating system',
    'ros2', 'behavior trees', 'motion planning', 'inverse kinematics',
    'reinforcement learning for robotics', 'sim-to-real transfer', 'domain randomization', 'mujoco',
    'isaac sim', 'pybullet', 'gaze estimation', 'trajectory optimization',
    'lidar data processing', 'radar fusion', 'sensor fusion', 'kalman filters',

    # --- AI Security, DevOps, & Cloud Architectures ---
    'owasp top 10 for llm', 'prompt injection protection', 'data poisoning defense', 'model inversion protection',
    'membership inference attacks', 'adversarial robustness', 'model extraction defense', 'secure enclave deployment',
    'confidential computing', 'iam roles for ai services', 'vulnerability scanning for ml containers', 'api gateway rate limiting',
    'input/output sanitization pipelines', 'aws sagemaker', 'azure ml studio', 'google vertex ai',
    'infrastructure as code', 'iac', 'terraform', 'pulumi',
    'cloudformation', 'helm charts', 'istio service mesh', 'ingress controllers',
    'multi-region deployment', 'hybrid cloud architecture', 'high availability', 'ha',
    'disaster recovery', 'dr', 'horizontal pod autoscaling', 'hpa',
    'vertical pod autoscaling', 'vpa', 'proof of concept', 'poc development',
    'enterprise software integration', 'legacy system modernization', 'custom api wrappers', 'client-facing technical delivery',
    'solutions scoping', 'value engineering', 'on-premise ai deployments', 'air-gapped environments',
    'saas delivery pipelines', 'technical stakeholder management',

    # --- Math, Foundations, & Advanced Technical Methodology ---
    'singular value decomposition', 'svd', 'principal component analysis', 'pca',
    'eigenvalues and eigenvectors', 'matrix factorization', 'jacobian matrix', 'hessian matrix',
    'stochastic gradient descent', 'sgd', 'adamw optimizer', 'rmsprop',
    'backpropagation through time', 'bptt', 'vanishing gradient problem', 'exploding gradient problem',
    'l1 and l2 regularization', 'dropout layers', 'batch normalization', 'layer normalization',
    'group normalization', 'bayesian inference', 'markov chain monte carlo', 'mcmc',
    'hidden markov models', 'hmm', 'gaussian mixture models', 'gmm',
    'maximum likelihood estimation', 'mle', 'maximum a posteriori', 'map',
    'p-value computation', 'a/b testing framework', 'multi-armed bandits', 'thompson sampling',
    'upper confidence bound', 'ucb', 'time series forecasting', 'arima',
    'sarimax', 'prophet', 'causal inference', 'propensity score matching',
    'synthetic controls', 'quantitative modeling', 'stochastic processes', 'cross-entropy loss',
    'mean squared error', 'mse', 'mean absolute error', 'mae',
    'huber loss', 'triplet loss', 'contrastive loss', 'dice loss',
    'focal loss', 'kl divergence', 'kullback-leibler', 'jensen-shannon divergence',
    'cosine embedding loss', 'constrained optimization', 'lagrange multipliers', 'convex optimization',
    'gradient clipping', 'learning rate schedules', 'cosine annealing',

    # --- Full-Stack AI, Frameworks, & Tooling ---
    'django rest framework', 'litestar', 'sanic', 'grpc',
    'graphql', 'apollo server', 'celery', 'redis',
    'memcached', 'pydantic', 'data validation', 'sqlalchemy',
    'tortoise-orm', 'motor', 'async mongodb', 'asyncio',
    'multiprocessing', 'threading', 'boto3', 'streamlit',
    'gradio', 'chainlit', 'dash', 'plotly',
    'shiny for python', 'react.js', 'next.js', 'vue.js',
    'typescript', 'tailwindcss', 'd3.js', 'three.js',
    'webgpu', 'webassembly', 'wasm', 'github actions',
    'gitlab ci/cd', 'jenkins', 'circleci', 'argo workflows',
    'argocd', 'sonarqube', 'pre-commit hooks', 'ruff',
    'black', 'mypy', 'type checking', 'pytest',
    'unittest', 'tox', 'poetry', 'dependency management',
    'pipenv', 'conda', 'venv',

    # --- Industry-Specific AI Domains ---
    'dicom imaging', 'fhir standards', 'alphafold', 'molecular property prediction',
    'electronic health records analytics', 'clinical natural language processing', 'bioinformatics pipelines', 'genomic sequencing data',
    'computed tomography segmentation', 'algorithmic trading systems', 'fraud detection engines', 'anti-money laundering models',
    'aml models', 'credit risk scoring', 'sentiment analysis for finance', 'high-frequency trading data',
    'hft data', 'market microstructure modeling', 'portfolio optimization models', 'hd mapping data',
    'end-to-end driving models', 'telemetry data processing', 'predictive maintenance algorithms', 'demand forecasting engines',
    'vehicle routing problem', 'vrp optimizers', 'warehouse automation systems', 'fleet logistics optimization'
]

def analyze_content_quality(transcript):
    """
    Heuristic content quality scorer:
      - Clarity   : avg sentence length (penalise extremes)
      - Relevance : technical keyword density
      - Structure : question/example/explanation indicators
      - Tech vocab: unique technical terms used
    Returns per-dimension scores + composite Content Quality Score.
    """
    text_lower = transcript.lower()
    words      = transcript.split()
    sentences  = [s.strip() for s in re.split(r'[.!?]+', transcript.strip()) if s.strip()]
    num_words  = max(len(words), 1)
    num_sent   = max(len(sentences), 1)

    # Clarity (ideal avg sentence 10-20 words)
    avg_sent_len = num_words / num_sent
    clarity = 90 if 10 <= avg_sent_len <= 20 else 70 if 7 <= avg_sent_len <= 25 else 50

    # Technical keyword detection
    detected = [kw for kw in TECH_KEYWORDS if kw in text_lower]
    unique_kw  = list(dict.fromkeys(kw.title() for kw in detected))  # preserve order, dedupe
    kw_density = len(unique_kw) / (num_words / 100)  # per 100 words
    relevance  = min(100, round(kw_density * 12))

    # Structure indicators
    structure_markers = [
        'first', 'second', 'third', 'finally', 'in conclusion',
        'for example', 'for instance', 'such as', 'because',
        'therefore', 'however', 'additionally', 'furthermore',
        'i have', 'i worked', 'i built', 'i developed',
        'my project', 'our team',
    ]
    structure_hits = sum(1 for m in structure_markers if m in text_lower)
    structure = min(100, structure_hits * 8 + 20)

    tech_vocab = min(100, len(unique_kw) * 7)

    content_quality = round(
        clarity    * 0.25 +
        relevance  * 0.30 +
        structure  * 0.25 +
        tech_vocab * 0.20
    )
    return {
        'clarity_score':    round(clarity),
        'relevance_score':  round(relevance),
        'structure_score':  round(structure),
        'tech_vocab_score': round(tech_vocab),
        'content_quality':  content_quality,
        'keywords':         unique_kw,
    }

In [18]:
# Section 18: Advanced Score Computation & Strengths/Improvements Generator
import numpy as np

def compute_advanced_scores(metrics, emotion_data, content_data, filler_data):
    """
    Produces a full score breakdown and weighted Overall Score.
    Weights (must sum to 1):
        Communication 0.25  Confidence 0.25  Content Quality 0.30
        Eye Contact 0.10    Emotion Stability 0.10
    """
    # Communication Score
    wpm          = metrics.get('wpm', 130)
    filler_r     = filler_data.get('filler_ratio', 5)
    wpm_score    = max(0, 100 - abs(wpm - 135) * 1.5)
    filler_score = max(0, 100 - filler_r * 8)
    communication = round(wpm_score * 0.45 + filler_score * 0.55, 1)

    # Confidence Score
    voice_conf  = metrics.get('voice_confidence', 75)
    eye_pct     = metrics.get('eye_contact', 0.5) * 100
    emotion_pos = (
        emotion_data['distribution'].get('happy',   0) +
        emotion_data['distribution'].get('neutral',  0)
    )
    confidence = round(voice_conf * 0.40 + eye_pct * 0.30 + emotion_pos * 0.30, 1)

    # Eye Contact Score
    eye_contact_score = round(min(100, eye_pct * 1.2), 1)

    if eye_pct >= 75:   eye_assessment = 'Excellent camera engagement throughout the interview.'
    elif eye_pct >= 50: eye_assessment = 'Good camera engagement throughout most of the interview.'
    elif eye_pct >= 30: eye_assessment = 'Moderate eye contact — try to look at the camera more consistently.'
    else:               eye_assessment = 'Low eye contact detected — focus on maintaining camera engagement.'

    content_quality   = content_data['content_quality']
    emotion_stability = emotion_data['stability_score']

    overall = round(
        communication     * 0.25 +
        confidence        * 0.25 +
        content_quality   * 0.30 +
        eye_contact_score * 0.10 +
        emotion_stability * 0.10,
        1
    )
    verdict = ('Excellent' if overall >= 90 else 'Good' if overall >= 75
               else 'Average' if overall >= 60 else 'Needs Improvement')

    return {
        'Communication Score':     communication,
        'Confidence Score':        confidence,
        'Eye Contact Score':       eye_contact_score,
        'Content Quality Score':   content_quality,
        'Emotion Stability Score': emotion_stability,
        'Overall Interview Score': overall,
        'Verdict':                 verdict,
        'Eye Assessment':          eye_assessment,
    }


def generate_strengths_improvements(advanced_scores, speaking_data, filler_data, content_data, emotion_data):
    """Automatically derive strengths and improvement areas."""
    strengths, improvements = [], []

    if speaking_data['pace'] == 'Ideal':
        strengths.append('✓ Good speaking speed (ideal pace)')
    else:
        improvements.append(f"Adjust speaking pace — currently {speaking_data['pace'].lower()}")

    if filler_data['severity'] in ('Excellent', 'Good'):
        strengths.append('✓ Minimal use of filler words')
    else:
        improvements.append('Reduce filler words (um, uh, like, etc.)')

    ec = advanced_scores['Eye Contact Score']
    if ec >= 70:
        strengths.append('✓ Good eye contact / camera engagement')
    else:
        improvements.append('Improve eye contact — look directly at the camera')

    tv = content_data['tech_vocab_score']
    if tv >= 50:
        strengths.append('✓ Strong technical vocabulary')
    else:
        improvements.append('Use more technical keywords relevant to your domain')

    if advanced_scores['Confidence Score'] >= 65:
        strengths.append('✓ Positive confidence level')
    else:
        improvements.append('Work on projecting confidence — voice modulation and posture')

    if content_data['structure_score'] >= 50:
        strengths.append('✓ Clear and structured communication')
    else:
        improvements.append('Improve answer structure — use STAR method (Situation, Task, Action, Result)')

    if emotion_data['stability_label'] in ('Excellent', 'Good'):
        strengths.append('✓ Stable emotional composure during the interview')
    else:
        improvements.append('Work on emotional regulation — practice mock interviews to reduce anxiety')

    if len(content_data['keywords']) >= 5:
        strengths.append(f"✓ Demonstrated knowledge of {len(content_data['keywords'])} technical concepts")
    else:
        improvements.append('Provide more specific technical examples and project references')

    return strengths, improvements

In [ ]:
# Section 19: Professional PDF Report Generation
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, HRFlowable
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.pagesizes import letter
from reportlab.lib import colors
from reportlab.lib.units import inch

def generate_professional_pdf(
    advanced_scores, speaking_data, filler_data, eye_contact_pct,
    emotion_data, content_data, strengths, improvements,
    transcript, duration_sec=0,
):
    """Multi-section professional PDF report. Saved as Interview_Professional_Report.pdf"""
    pdf_path = 'Interview_Professional_Report.pdf'
    doc = SimpleDocTemplate(pdf_path, pagesize=letter,
                            leftMargin=0.75*inch, rightMargin=0.75*inch,
                            topMargin=0.75*inch,  bottomMargin=0.75*inch)
    styles = getSampleStyleSheet()
    story  = []

    # Custom styles
    h1   = ParagraphStyle('H1', parent=styles['Heading1'], fontSize=18, spaceAfter=6, textColor=colors.HexColor('#003366'))
    h2   = ParagraphStyle('H2', parent=styles['Heading2'], fontSize=13, spaceAfter=4, textColor=colors.HexColor('#005599'))
    body = styles['Normal']
    bold = ParagraphStyle('bold', parent=body, fontName='Helvetica-Bold')
    small= ParagraphStyle('small', parent=body, fontSize=9)

    def section(title):
        story.append(Spacer(1, 10))
        story.append(HRFlowable(width='100%', thickness=1, color=colors.HexColor('#005599')))
        story.append(Paragraph(title, h2))

    def kv(key, value):
        story.append(Paragraph(f'<b>{key}:</b>  {value}', body))

    def rule():
        story.append(Spacer(1, 4))
        story.append(HRFlowable(width='100%', thickness=0.5, color=colors.lightgrey))
        story.append(Spacer(1, 4))

    # Title
    story.append(Paragraph('🎯 AI Interview Analysis Report', h1))
    story.append(Paragraph('Generated by AI Interview Analyzer · Powered by DeepFace, Whisper & Transformers', small))
    story.append(Spacer(1, 12))

    # 1. Candidate Summary
    section('1. Candidate Summary')
    dur_min = int(duration_sec // 60)
    dur_sec = int(duration_sec %  60)
    overall = advanced_scores['Overall Interview Score']
    verdict = advanced_scores['Verdict']
    kv('Interview Duration',  f'{dur_min}m {dur_sec}s')
    kv('Total Words Spoken',  speaking_data['total_words'])
    kv('Overall Score',       f'{overall:.1f} / 100')
    kv('Interview Verdict',   verdict)

    # 2. Score Breakdown
    section('2. Score Breakdown')
    score_data = [
        ['Category',          'Score'],
        ['Communication',     f"{advanced_scores['Communication Score']:.1f}"],
        ['Confidence',        f"{advanced_scores['Confidence Score']:.1f}"],
        ['Eye Contact',       f"{advanced_scores['Eye Contact Score']:.1f}"],
        ['Content Quality',   f"{advanced_scores['Content Quality Score']:.1f}"],
        ['Emotion Stability', f"{advanced_scores['Emotion Stability Score']:.1f}"],
        ['OVERALL',           f'{overall:.1f}'],
    ]
    def score_colour(val):
        v = float(val)
        return (colors.HexColor('#d4edda') if v >= 75 else
                colors.HexColor('#fff3cd') if v >= 60 else
                colors.HexColor('#f8d7da'))

    table_style = TableStyle([
        ('TEXTCOLOR',      (0,0),  (-1,0),  colors.white),
        ('FONTNAME',       (0,0),  (-1,0),  'Helvetica-Bold'),
        ('BACKGROUND',     (0,0),  (-1,0),  colors.HexColor('#003366')),
        ('ALIGN',          (0,0),  (-1,-1), 'CENTER'),
        ('FONTNAME',       (0,-1), (-1,-1), 'Helvetica-Bold'),
        ('BACKGROUND',     (0,-1), (-1,-1), colors.HexColor('#005599')),
        ('TEXTCOLOR',      (0,-1), (-1,-1), colors.white),
        ('GRID',           (0,0),  (-1,-1), 0.5, colors.grey),
        ('ROWBACKGROUNDS', (0,1),  (-1,-2), [colors.whitesmoke, colors.white]),
    ])
    tbl = Table(score_data, colWidths=[3.5*inch, 2.5*inch])
    tbl.setStyle(table_style)
    story.append(tbl)

    # 3. Speaking Analysis
    section('3. Speaking Analysis')
    kv('Total Words Spoken',     speaking_data['total_words'])
    kv('Words Per Minute (WPM)', f"{speaking_data['wpm']:.1f}")
    kv('Pace Classification',    speaking_data['pace'])
    story.append(Paragraph(speaking_data['pace_explanation'], body))

    # 4. Filler Word Analysis
    section('4. Filler Word Analysis')
    kv('Total Filler Words', filler_data['total_fillers'])
    kv('Filler Ratio',       f"{filler_data['filler_ratio']:.1f}%")
    kv('Severity Level',     filler_data['severity'])
    if filler_data['common_fillers']:
        top = ', '.join(f'{w} ({c}x)' for w, c in filler_data['common_fillers'])
        kv('Most Common Fillers', top)

    # 5. Eye Contact Analysis
    section('5. Eye Contact Analysis')
    kv('Eye Contact Percentage', f"{eye_contact_pct*100:.1f}%")
    kv('Eye Contact Score',      f"{advanced_scores['Eye Contact Score']:.1f}")
    story.append(Paragraph(advanced_scores['Eye Assessment'], body))

    # 6. Emotion Stability Analysis
    section('6. Emotion Stability Analysis')
    kv('Stability Score', f"{emotion_data['stability_score']:.1f}")
    kv('Stability Level', emotion_data['stability_label'])
    rule()
    story.append(Paragraph('<b>Emotion Distribution:</b>', bold))
    for emo, pct in emotion_data['distribution'].items():
        if pct > 0:
            story.append(Paragraph(f'  • {emo.title()}: {pct:.1f}%', body))

    # 7. Content Quality Analysis
    section('7. Content Quality Analysis')
    kv('Clarity Score',    content_data['clarity_score'])
    kv('Relevance Score',  content_data['relevance_score'])
    kv('Structure Score',  content_data['structure_score'])
    kv('Tech Vocab Score', content_data['tech_vocab_score'])
    kv('Content Quality',  content_data['content_quality'])

    if content_data['keywords']:
        section('8. Technical Keywords Detected')
        story.append(Paragraph(', '.join(content_data['keywords'][:15]), body))

    # 9. Strengths
    section('9. Strengths')
    for s in strengths:
        story.append(Paragraph(s, body))

    # 10. Areas for Improvement
    section('10. Areas for Improvement')
    for imp in improvements:
        story.append(Paragraph(f'⚠  {imp}', body))

    # 11. Transcript Summary
    section('11. Transcript (First 1000 Characters)')
    story.append(Paragraph(transcript[:20000] + ' …', small))

    doc.build(story)
    return pdf_path